# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guided template for loading, exploring, and processing the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL for the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

For each record set defined in the metadata, examine the fields and their `@id`s.

In [ ]:
# List available record sets and their @ids
if hasattr(metadata, 'recordSets'):
    record_sets = metadata.recordSets
else:
    record_sets = metadata.recordSet
    # recordSet might be a list or a single item
    if isinstance(record_sets, dict):
        record_sets = [record_sets]

print('Available Record Sets:')
record_set_ids = []
for rs in record_sets:
    print(f"- Name: {getattr(rs, 'name', 'N/A')} (@id={rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)})")
    record_set_ids.append(rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None))

    # List fields in each record set
    fields = getattr(rs, 'fields', None) or getattr(rs, 'field', None)
    if fields is not None:
        print('  Fields:')
        for f in fields:
            print(f"    - {getattr(f, 'name', 'N/A')} (@id={f['@id'] if isinstance(f, dict) else getattr(f, '@id', None)})")
    else:
        print('  No fields listed.')

# Preview records for a record set
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"\nPreviewing records for record set @id={preview_id}:")
    for i, rec in enumerate(dataset.records(record_set=preview_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. Use each record set's `@id`, and reference all columns and fields by their `@id`s.

In [ ]:
# Extract and load ALL record sets to DataFrames
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns in record set {rs_id}: {df.columns.tolist()}")
    else:
        print(f"No records found for record set {rs_id}.")

# Select the main record set for detailed analysis
main_rs_id = record_set_ids[0] if record_set_ids else None

if main_rs_id:
    print(f"\nSample of main record set (@id: {main_rs_id}):")
    display_cols = dataframes[main_rs_id].columns[:8]
    print(dataframes[main_rs_id][display_cols].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using columns referenced by their `@id`. Demonstrate filtering, normalization, and grouping.

In [ ]:
# Exploratory data analysis on the main record set
df = dataframes[main_rs_id]

# Identify numeric fields by examining column names and their @ids
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or col.lower().startswith('phq') or col.lower().startswith('gad') or col.lower().startswith('psq')]
print(f"Numeric candidate fields: {numeric_candidates}")

# Choose a numeric field @id (e.g., PHQ-9 score)
# If actual @id is available from the overview, replace below. Here, we use column name as the @id.
numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]
print(f"Analyzing numeric field: {numeric_field_id}")

# Filter outliers and normalize
threshold = df[numeric_field_id].mean() + 2*df[numeric_field_id].std() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize using z-score
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a demographic field (@id) (e.g., 'gender', 'marital_status', etc.)
potential_group_fields = ['gender', 'marital_status', 'village', 'level_of_education']
group_field_id = None
for gf in potential_group_fields:
    if gf in df.columns:
        group_field_id = gf
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (@id: {group_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize distributions and relationships for fields using their `@id`s.

In [ ]:
# Visualize numeric field distribution
plt.figure(figsize=(8,5))
df[numeric_field_id].hist(bins=20)
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

# If grouping field is available, visualize its relationship with numeric scores
if group_field_id:
    plt.figure(figsize=(8,6))
    grouped_df.plot.bar(x=group_field_id, y=numeric_field_id, legend=False, ax=plt.gca())
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This exploration demonstrates loading and analysis of the FAIR² Mental Health Survey Dataset using `mlcroissant`.

- We loaded metadata and record sets using the Croissant schema URL.
- We extracted data referencing all entities by their `@id`.
- Numeric mental health assessment fields and demographic fields were used for basic EDA.
- Data was visualized to highlight patterns and possible relationships.

For further analysis, refer to the original Croissant schema and full documentation of the dataset.